Extraction of data from Zenodo for affiliation and ORCID

In [ ]:
import requests
import json
import time
import pandas as pd
from pathlib import Path

# loading IRIS to get UNIBO researchers ORCID IDs
Iris_df = pd.read_csv(r"C:\Users\pietr\Desktop\OpenScience\ODS_L1_IR_ITEM_CON_PERSON.csv", encoding="utf-8")
unibo_orcids = Iris_df["ORCID"].dropna().unique().tolist()

base_url = "https://zenodo.org/api/records"

# Search parameters with query for UNIBO
params = {
    'q': 'contributors.affiliation:"University of Bologna" OR contributors.affiliation:"Università di Bologna" OR '
         'contributors.affiliation:"UNIBO" OR contributors.affiliation:"Alma Mater Studiorum" OR '
         'creators.affiliation:"University of Bologna" OR creators.affiliation:"Università di Bologna" OR '
         'creators.affiliation:"UNIBO" OR creators.affiliation:"Alma Mater Studiorum"',
    'size': 100,  
    'page': 1,    
}

all_records = []
orcid_records = []
total_records = 0
processed_records = 0

continue_pagination = True

try:
    # Get records with Bologna in the affiliation
    while continue_pagination:
       
        response = requests.get(base_url, params=params)
        response.raise_for_status()  # Raise exception for HTTP errors
        data = response.json()
        
        # Get the hits from this page
        hits = data.get('hits', {}).get('hits', [])
        
        # Update total if this is the first page
        if params['page'] == 1:
            total_records = data.get('hits', {}).get('total', 0)
            print(f"Total records found with UNIBO affiliation: {total_records}")
        
        all_records.extend(hits)
        processed_records += len(hits)
        
        if len(hits) > 0 and processed_records < total_records:
            params['page'] += 1
            time.sleep(1)  
        else:
            continue_pagination = False
    
    print(f"Found {len(all_records)} records with UNIBO affiliation.")
    
    # Searching for records with UNIBO ORCID IDs
    if unibo_orcids:
        print(f"Searching {len(unibo_orcids)} ORCID IDs ")
        
        batch_size = 10  # How many ORCIDs to query at once
        
        total_orcids = len(unibo_orcids)
        total_batches = (total_orcids + batch_size - 1) // batch_size

        for batch_start in range(0, total_orcids, batch_size):
            batch_end = min(batch_start + batch_size, total_orcids)
            batch_orcids = unibo_orcids[batch_start:batch_end]
            
            # Generate ORCID query part for this batch
            orcid_queries = []
            for orcid in batch_orcids:
                # Make sure the ORCID is a string and properly formatted
                if pd.notna(orcid):
                    orcid = str(orcid).strip()
                    orcid_queries.append(f'creators.orcid:"{orcid}" OR contributors.orcid:"{orcid}"')
                
            # Exclude UNIBP affiliations 
            orcid_query = " OR ".join(orcid_queries)
            orcid_params = {
                'q': f'({orcid_query}) AND NOT (contributors.affiliation:"University of Bologna" OR contributors.affiliation:"Università di Bologna" OR '
                    'contributors.affiliation:"UNIBO" OR contributors.affiliation:"Alma Mater Studiorum" OR '
                    'creators.affiliation:"University of Bologna" OR creators.affiliation:"Università di Bologna" OR '
                    'creators.affiliation:"UNIBO" OR creators.affiliation:"Alma Mater Studiorum")',
                'size': 100,
                'page': 1,
            }
            
            # Reset pagination variables for this batch
            batch_continue = True
            orcid_total = 0
            orcid_processed = 0
            
            while batch_continue:
                try:

                    response = requests.get(base_url, params=orcid_params)
                    response.raise_for_status()
                    data = response.json()
                    
                    hits = data.get('hits', {}).get('hits', [])
                    
                    # Update total for this batch if this is the first page
                    if orcid_params['page'] == 1:
                        orcid_total = data.get('hits', {}).get('total', 0)
                        print(f" Records found for this ORCID batch: {orcid_total}")
                    
                    orcid_records.extend(hits)
                    orcid_processed += len(hits)
                    
                    # Check if we should continue to the next page for this batch
                    if len(hits) > 0 and orcid_processed < orcid_total:
                        orcid_params['page'] += 1
                        time.sleep(1)  
                    else:
                        batch_continue = False
                        
                except Exception as e:
                    print(f" Error: {e}")
                    batch_continue = False 

            time.sleep(2)
        
        print(f"Extracted {len(orcid_records)} additional records with ORCIDs.")
        
        all_records.extend(orcid_records)
    
    
    output_file = Path("ZenodoData2.json")
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(all_records, f, indent=2, ensure_ascii=False)
    
    print(f"\nData extraction complete {len(all_records)} total records saved to {output_file}")
    
except Exception as e:
    print(f"Error: {e}")

Precision calculation

In [1]:
import json
import random
from pathlib import Path

input_file = Path(r"C:\Users\pietr\Desktop\github\2024-2025\crisis\code\zenodo\ZenodoData.json")
output_file = Path(r"C:\Users\pietr\Desktop\github\2024-2025\crisis\code\zenodo\ZenodoData_sample100.json")

with open(input_file, "r", encoding="utf-8") as f:
    all_records = json.load(f)

print(f"Total records in file: {len(all_records)}")

sample = random.sample(all_records, min(100, len(all_records)))

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(sample, f, indent=2, ensure_ascii=False)

print(f"Saved {len(sample)} random records to {output_file}")

Total records in file: 5434
Saved 100 random records to C:\Users\pietr\Desktop\github\2024-2025\crisis\code\zenodo\ZenodoData_sample100.json


In [3]:
import json
import csv
from pathlib import Path

input_file = Path(r"C:\Users\pietr\Desktop\github\2024-2025\crisis\code\zenodo\ZenodoData_sample100.json")
output_file = Path(r"C:\Users\pietr\Desktop\github\2024-2025\crisis\code\zenodo\ZenodoData_affiliation_check.csv")

UNIBO_STRINGS = [
    "university of bologna",
    "università di bologna",
    "unibo",
    "alma mater studiorum"
]

def has_unibo_affiliation(record):
    metadata = record.get("metadata", {})
    for group in ["creators", "contributors"]:
        for person in metadata.get(group, []):
            affiliation = person.get("affiliation", "")
            if affiliation and any(s in affiliation.lower() for s in UNIBO_STRINGS):
                return True
    return False

with open(input_file, "r", encoding="utf-8") as f:
    records = json.load(f)

tagged_count = 0
unibo_records = []
for record in records:
    has_affiliation = has_unibo_affiliation(record)
    if has_affiliation:
        tagged_count += 1
        unibo_records.append(record)
    print(f"ID: {record['id']} | Title: {record['metadata']['title'][:60]} | UNIBO affiliation: {has_affiliation}")

print(f"\nTotal records: {len(records)}")
print(f"With UNIBO affiliation string: {tagged_count}")
print(f"Without UNIBO affiliation string (ORCID only): {len(records) - tagged_count}")

with open(output_file, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "title", "doi", "publication_date", "zenodo_url", "is_correct"])
    for record in unibo_records:
        meta = record.get("metadata", {})
        writer.writerow([
            record.get("id"),
            meta.get("title"),
            meta.get("doi"),
            meta.get("publication_date"),
            record.get("links", {}).get("self_html"),
            ""
        ])

print(f"\nSaved {len(unibo_records)} records to {output_file}")

ID: 10925257 | Title: The ACTonHEART Study: a Randomized Controlled Clinical Trial | UNIBO affiliation: True
ID: 4993847 | Title: Data from: Charting the NF-κB pathway interactome map | UNIBO affiliation: True
ID: 14329953 | Title: Letter by Gasparini and Boriani Regarding Article, "Cardiac  | UNIBO affiliation: True
ID: 5463938 | Title: Whole Genome Sequencing Prioritizes CHEK2, EWSR1, and TIAM1  | UNIBO affiliation: True
ID: 10102898 | Title: Biomarkers of neurodegeneration in isolated and antidepressa | UNIBO affiliation: False
ID: 15039051 | Title: Data set for "Total Least Squares In-Field Identification fo | UNIBO affiliation: False
ID: 4633475 | Title: Displacement time series from GNSS stations in Central Italy | UNIBO affiliation: True
ID: 13981406 | Title: Highly Efficient Photoanodic Material: Utilizing Dihydrolipo | UNIBO affiliation: True
ID: 10697664 | Title: Inverse opals with reactive surface chemistry as sensors for | UNIBO affiliation: False
ID: 5070288 | Title: Datas

Category count

In [ ]:
import pandas as pd

input_file = r'C:\Users\pietr\Downloads\mashup_IRIS_subset_v3.csv'
df = pd.read_csv(input_file)


zenodo_records = df[df['src_repo'].str.lower() == 'zenodo'].copy()

print(f"Total Zenodo records found: {len(zenodo_records)}")
display(zenodo_records.head())


output_file = 'zenodo_only_records.csv'
zenodo_records.to_csv(output_file, index=False)

print(f"Successfully saved Zenodo records to {output_file}")

Total Zenodo records found: 2522


,title,id,doi,creators,orcid,date,description,resource_type,url,type,rights,publisher,relation,communities,swh_id,keywords,src_repo,issn,pmid,iris_cat
0,Footactile rhythmics: protocols and data colle...,5504259.0,10.5281/zenodo.5504259,"[""Dall'Osso, Giorgio""]",['0000-0002-4219-7513'],2021-09-13,The data shared refer to research investigatin...,dataset,https://doi.org/10.5281/zenodo.5504259,dataset,open,Alma Mater Studiorum - Università di Bologna,['5504258'],[],NaN,"['haptic, protocol, design, advanced design, b...",zenodo,NaN,NaN,7.05 Banche dati
1,Addressing the Challenges of Health Data Stand...,15358180.0,10.5281/zenodo.15358180,"['Marfoglia, Alberto', 'Arcobelli, Valerio Ant...","['0009-0000-5857-2376', '0000-0002-1262-9899',...",2025-05-07,This table presents the data extraction from t...,dataset,https://doi.org/10.5281/zenodo.15358180,dataset,open,zenodo,['15358179'],[],NaN,"['Health Data Standard', 'FHIR', 'OMOP-CDM', '...",zenodo,NaN,NaN,7.05 Banche dati
2,davidedomini/experiments-2025-acsos-marl-for-s...,15249055.0,10.5281/zenodo.15249055,"['Filippo Venturini', 'Davide Domini', 'Semant...","['zenodo', 'zenodo', 'zenodo', 'zenodo']",2025-04-19,1.2.5 (2025-04-19)\nBug Fixes\n\ncreate data f...,software,https://doi.org/10.5281/zenodo.15249055,software,open,zenodo,['14938112'],[],swh:1:dir:ef46868654a0f173b560eba251fbfdff6e30...,NaN,zenodo,NaN,NaN,7.04 Software
3,AlchemistSimulator/Alchemist: 42.2.0,15411447.0,10.5281/zenodo.15411447,"['Danilo Pianini', 'Vuksaa', 'Mend Renovate', ...","['zenodo', 'zenodo', 'zenodo', 'zenodo', 'zeno...",2025-05-14,42.2.0 (2025-05-14)\nFeatures\n\nloading: add ...,software,https://doi.org/10.5281/zenodo.15411447,software,open,zenodo,['8146632'],[],NaN,NaN,zenodo,NaN,NaN,7.04 Software
4,Synthesis and characterization of maleimide-ba...,15261097.0,10.5281/zenodo.15261097,"['Kampasis, Dionysis']",['0009-0004-8554-8936'],2025-04-22,The maleimide-based compounds described herein...,dataset,https://doi.org/10.5281/zenodo.15261097,dataset,embargoed,zenodo,['15261096'],[],NaN,"['Medicinal Chemistry', 'Multi-Target-Directed...",zenodo,NaN,NaN,7.05 Banche dati


Successfully saved Zenodo records to zenodo_only_records.csv


In [ ]:
import pandas as pd


input_file = r'C:\Users\pietr\Desktop\github\2024-2025\crisis\code\zenodo\zenodo_only_records.csv'

df = pd.read_csv(input_file)


total_records = len(df)
cat_counts = df['iris_cat'].value_counts()
cat_percentages = (cat_counts / total_records) * 100


summary_df = pd.DataFrame({
    'Records': cat_counts,
    'Percentage (%)': cat_percentages
})


print("="*60)
print(f"ZENODO FILE ANALYSIS")
print(f"Path: {input_file}")
print("="*60)
print(f"Total entries in file: {total_records}")
print("-"*60)
print("Breakdown per Category (iris_cat):")
print("-"*60)


display(summary_df.style.format({'Percentage (%)': '{:.2f}%'}))

print("="*60)


ZENODO FILE ANALYSIS
Path: C:\Users\pietr\Desktop\github\2024-2025\crisis\code\zenodo\zenodo_only_records.csv
Total entries in file: 2522
------------------------------------------------------------
Breakdown per Category (iris_cat):
------------------------------------------------------------


,Records,Percentage (%)
iris_cat,,
7.05 Banche dati,1275,50.56%
7.13 Rapporto tecnico,449,17.80%
7.04 Software,375,14.87%
7.07 Disegno,371,14.71%
7.03 Prodotto dell’ingegneria civile e dell’architettura,27,1.07%
7.06 Composizione musicale,21,0.83%
7.09 Performance,4,0.16%
